# Smart Inventory & Waste Reducer — ML Notebook
### Model Comparison (GradientBoosting / RandomForest / MLP) + Pure NumPy LSTM

Notebook ini regenerasi dari `Smart_Inventory_Model_Comparison.ipynb` +
`smart_inventory_lstm_colab.py` versi sebelumnya — sekarang disinkronkan
dengan `ml/app.py` (versi production, 1-file) supaya artefak yang dihasilkan
di sini (`gb_demand.pkl`, `gb_profit.pkl`, `feat_mean.npy`, dst) **kompatibel
langsung** dengan Flask API yang live, tanpa mismatch jumlah fitur.

**Cara pakai di Google Colab:**
1. Upload `inventory_dummy_10k.csv` (Cell 2 di bawah ada tombol upload-nya)
2. Run semua cell berurutan (Runtime → Run all)
3. Di akhir notebook, download `gb_demand.pkl` / `gb_profit.pkl` / `feat_*.npy`
   / `lstm_weights.npz`, lalu taruh di folder `ml/` project kamu

**Waktu eksekusi total:** ±4-6 menit (GB ±2 menit, RF+MLP pembanding ±1 menit,
LSTM ±2 menit). Bisa dipercepat dengan mengecilkan `N_EPOCHS_LSTM` di Cell 8.

## Cell 1 — Setup & Import

In [ ]:
!pip install scikit-learn matplotlib seaborn pandas numpy flask flask-cors -q

import time, warnings, pickle, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

COLORS = {'gb': '#2563eb', 'rf': '#16a34a', 'mlp': '#dc2626', 'lstm': '#9333ea', 'ma': '#f59e0b'}
CAT_COLORS = ['#3b82f6', '#22c55e', '#f59e0b', '#ef4444', '#8b5cf6', '#14b8a6', '#f97316']

print('✅ Setup selesai')
print(f'   NumPy {np.__version__}  |  Pandas {pd.__version__}')

## Cell 2 — Load Data

Jalankan baris upload di bawah kalau di Colab & belum ada file-nya. Kalau
notebook ini dijalankan di folder yang sama dengan `inventory_dummy_10k.csv`
(mis. clone dari repo project), baris upload akan otomatis dilewati.

In [ ]:
CSV_PATH = 'inventory_dummy_10k.csv'

if not os.path.exists(CSV_PATH):
    try:
        from google.colab import files  # noqa
        print('📤 Upload inventory_dummy_10k.csv:')
        uploaded = files.upload()
        CSV_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"'{CSV_PATH}' tidak ditemukan. Taruh file CSV di folder yang sama "
            "dengan notebook ini, atau jalankan di Google Colab untuk upload manual."
        )

df = pd.read_csv(CSV_PATH)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Product_Name', 'Date']).reset_index(drop=True)

print(f"✅ Loaded {len(df):,} baris · {df['Product_Name'].nunique()} produk · {df['Category'].nunique()} kategori")
print(f"   Rentang: {df['Date'].min().date()} → {df['Date'].max().date()}")
df.head(3)

## Cell 3 — EDA Singkat

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle('EDA — Smart Inventory Dataset', fontsize=14, fontweight='bold')

ax = axes[0]
for i, cat in enumerate(df['Category'].unique()):
    ax.hist(df[df['Category'] == cat]['Actual_Demand'], bins=25, alpha=0.55,
            color=CAT_COLORS[i % len(CAT_COLORS)], label=cat, density=True)
ax.set_title('Distribusi Demand per Kategori', fontweight='bold', fontsize=11)
ax.set_xlabel('Units/hari'); ax.legend(fontsize=7)

monthly = df.groupby(df['Date'].dt.to_period('M')).agg(
    Revenue=('Revenue', 'sum'), NetProfit=('Net_Profit', 'sum'), Waste=('Waste_Value', 'sum')
).reset_index()
monthly['dt'] = monthly['Date'].dt.to_timestamp()
ax = axes[1]
ax.fill_between(monthly['dt'], monthly['Revenue'] / 1e3, alpha=0.25, color='#2563eb', label='Revenue')
ax.fill_between(monthly['dt'], monthly['NetProfit'] / 1e3, alpha=0.6, color='#16a34a', label='Net Profit')
ax.fill_between(monthly['dt'], -monthly['Waste'] / 1e3, alpha=0.5, color='#ef4444', label='Waste Loss')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Revenue vs Profit vs Waste / Bulan', fontweight='bold', fontsize=11)
ax.set_ylabel('USD (ribuan)'); ax.legend(fontsize=8)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=40, ha='right')

pivot = df.pivot_table(values='Actual_Demand', index='DayOfWeek', columns='Month', aggfunc='mean')
pivot.index = ['Sen', 'Sel', 'Rab', 'Kam', 'Jum', 'Sab', 'Min']
sns.heatmap(pivot, ax=axes[2], cmap='YlOrRd', cbar_kws={'label': 'Avg Demand'})
axes[2].set_title('Demand: Hari × Bulan', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('01_eda_overview.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ EDA → 01_eda_overview.png')

## Cell 4 — Feature Engineering (33 fitur)

**Penting:** urutan & definisi 33 fitur ini harus identik dengan
`FEATURES`/`engineer_features()` di `ml/app.py`, supaya model yang dilatih
di notebook ini bisa langsung dipakai oleh Flask API tanpa retrain ulang.

In [ ]:
for sh in [1, 2, 3, 7, 14, 21, 30]:
    df[f'demand_lag{sh}'] = df.groupby('Product_Name')['Actual_Demand'].shift(sh)
for w in [3, 5, 7, 14, 21, 30]:
    df[f'demand_roll{w}'] = df.groupby('Product_Name')['Actual_Demand'].transform(
        lambda x: x.rolling(w, min_periods=1).mean())
df['demand_std7'] = df.groupby('Product_Name')['Actual_Demand'].transform(
    lambda x: x.rolling(7, min_periods=1).std().fillna(0))
df['demand_min7'] = df.groupby('Product_Name')['Actual_Demand'].transform(
    lambda x: x.rolling(7, min_periods=1).min())
df['demand_max7'] = df.groupby('Product_Name')['Actual_Demand'].transform(
    lambda x: x.rolling(7, min_periods=1).max())
df['demand_trend7'] = df['demand_roll3'] - df['demand_roll14']
df['demand_momentum'] = df.groupby('Product_Name')['Actual_Demand'].transform(
    lambda x: x.rolling(3, min_periods=1).mean() - x.rolling(7, min_periods=1).mean())
df['profit_roll7'] = df.groupby('Product_Name')['Net_Profit'].transform(
    lambda x: x.rolling(7, min_periods=1).mean())
df['gross_roll7'] = df.groupby('Product_Name')['Gross_Profit'].transform(
    lambda x: x.rolling(7, min_periods=1).mean())
df['waste_rate'] = df['Waste_Units'] / (df['Stock_Level'] + 1)
df['price_ratio'] = df['Unit_Price'] / (df['Cost_Price'] + 0.01)
df['quarter'] = df['Month'].apply(lambda m: (m - 1) // 3 + 1)

df = df.dropna(subset=['demand_lag7', 'demand_lag14', 'demand_lag30']).reset_index(drop=True)

FEATURES = [
    'Month', 'DayOfWeek', 'DayOfYear', 'Weekend', 'Seasonal_Factor', 'quarter',
    'Fill_Level_Pct', 'Stock_Level', 'Unit_Price', 'Cost_Price', 'price_ratio', 'Base_Demand',
    'demand_lag1', 'demand_lag2', 'demand_lag3', 'demand_lag7', 'demand_lag14',
    'demand_lag21', 'demand_lag30',
    'demand_roll3', 'demand_roll5', 'demand_roll7', 'demand_roll14',
    'demand_roll21', 'demand_roll30',
    'demand_std7', 'demand_min7', 'demand_max7',
    'demand_trend7', 'demand_momentum',
    'profit_roll7', 'gross_roll7', 'waste_rate',
]

X  = df[FEATURES].values.astype(np.float32)
y  = df['Actual_Demand'].values.astype(np.float32)
yg = df['Gross_Profit'].values.astype(np.float32)

X_MEAN = X.mean(0).astype(np.float64)
X_STD  = (X.std(0) + 1e-8).astype(np.float64)

SPLIT = int(len(X) * 0.85)
X_tr, X_te   = X[:SPLIT], X[SPLIT:]
y_tr, y_te   = y[:SPLIT], y[SPLIT:]
yg_tr, yg_te = yg[:SPLIT], yg[SPLIT:]

print(f'✅ Features : {len(FEATURES)} (cek: harus 33 → {"OK" if len(FEATURES)==33 else "MISMATCH!"})')
print(f'   Train    : {len(X_tr):,}  |  Test : {len(X_te):,}')

## Cell 5 — Train & Bandingkan Model

Model utama yang di-deploy adalah **Gradient Boosting** (dipakai live oleh
Flask API). Random Forest, MLP, dan Moving-Average baseline dilatih sebagai
pembanding untuk menunjukkan kenapa GB dipilih.

In [ ]:
def metrics(pred, actual, name=''):
    mape = np.mean(np.abs((actual - pred) / (np.abs(actual) + 1e-6))) * 100
    return {'name': name, 'acc': max(0.0, 100 - mape), 'mape': mape,
            'mae': mean_absolute_error(actual, pred),
            'rmse': np.sqrt(mean_squared_error(actual, pred)),
            'r2': r2_score(actual, pred)}

GB_PARAMS = dict(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.85,
                  min_samples_leaf=10, random_state=42, validation_fraction=0.1,
                  n_iter_no_change=15, tol=1e-4)

results = {}
print('[1/4] Gradient Boosting (demand — model yang di-deploy)...')
t0 = time.time()
gb_demand = GradientBoostingRegressor(**GB_PARAMS)
gb_demand.fit(X_tr, y_tr)
results['gb'] = metrics(gb_demand.predict(X_te), y_te, 'Gradient Boosting')
results['gb']['time'] = time.time() - t0
print(f"   ✅ acc={results['gb']['acc']:.1f}%  MAPE={results['gb']['mape']:.1f}%  ({results['gb']['time']:.0f}s)")

print('[2/4] Gradient Boosting (gross profit)...')
gb_profit = GradientBoostingRegressor(**GB_PARAMS)
gb_profit.fit(X_tr, yg_tr)
res_profit = metrics(gb_profit.predict(X_te), yg_te, 'GB Gross Profit')
print(f"   ✅ acc={res_profit['acc']:.1f}%  MAE=${res_profit['mae']:.2f}")

print('[3/4] Random Forest (pembanding)...')
t0 = time.time()
rf = RandomForestRegressor(n_estimators=200, max_depth=14, min_samples_leaf=8, n_jobs=-1, random_state=42)
rf.fit(X_tr, y_tr)
results['rf'] = metrics(rf.predict(X_te), y_te, 'Random Forest')
results['rf']['time'] = time.time() - t0
print(f"   ✅ acc={results['rf']['acc']:.1f}%  MAPE={results['rf']['mape']:.1f}%  ({results['rf']['time']:.0f}s)")

print('[4/4] MLP Neural Network (pembanding)...')
t0 = time.time()
mlp = MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam',
                    learning_rate_init=0.001, max_iter=200, early_stopping=True,
                    validation_fraction=0.1, n_iter_no_change=12, random_state=42, batch_size=512)
mlp.fit(X_tr, y_tr)
results['mlp'] = metrics(mlp.predict(X_te), y_te, 'MLP (128-64-32)')
results['mlp']['time'] = time.time() - t0
print(f"   ✅ acc={results['mlp']['acc']:.1f}%  MAPE={results['mlp']['mape']:.1f}%  ({results['mlp']['time']:.0f}s)")

pred_ma = df.iloc[SPLIT:SPLIT + len(y_te)]['demand_roll7'].values
results['ma'] = metrics(pred_ma, y_te, 'Moving Average (7d)')
results['ma']['time'] = 0

print('\n' + '═' * 64)
print(f"  {'Model':<22} {'Accuracy':>9} {'MAPE':>8} {'R²':>7}")
for k, r in sorted(results.items(), key=lambda x: -x[1]['acc']):
    star = ' 🏆' if r['acc'] == max(v['acc'] for v in results.values()) else ''
    print(f"  {r['name']:<22} {r['acc']:>8.1f}% {r['mape']:>7.1f}% {r['r2']:>7.3f}{star}")
print('═' * 64)

## Cell 6 — Visualisasi Perbandingan

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle('Perbandingan Model — Demand Forecasting', fontsize=14, fontweight='bold')

N_SHOW = 150
ax = axes[0]
ax.plot(y_te[-N_SHOW:], color='#1e293b', lw=2, label='Actual', zorder=10)
for k, col in zip(['gb', 'rf', 'mlp', 'ma'], [COLORS['gb'], COLORS['rf'], COLORS['mlp'], COLORS['ma']]):
    p = results[k]['pred'] if 'pred' in results[k] else None
ax.plot(gb_demand.predict(X_te)[-N_SHOW:], color=COLORS['gb'], lw=1.4, alpha=0.85,
        label=f"GB ({results['gb']['acc']:.1f}%)")
ax.plot(rf.predict(X_te)[-N_SHOW:], color=COLORS['rf'], lw=1.2, alpha=0.75, linestyle='--',
        label=f"RF ({results['rf']['acc']:.1f}%)")
ax.set_title(f'Actual vs Predicted — {N_SHOW} Hari Terakhir', fontweight='bold', fontsize=11)
ax.legend(fontsize=8)

ax = axes[1]
names = [results[k]['name'] for k in ['gb', 'rf', 'mlp', 'ma']]
accs  = [results[k]['acc'] for k in ['gb', 'rf', 'mlp', 'ma']]
bars = ax.bar(names, accs, color=[COLORS['gb'], COLORS['rf'], COLORS['mlp'], COLORS['ma']], alpha=0.85)
ax.set_ylim(0, 105); ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy Comparison', fontweight='bold', fontsize=11)
ax.axhline(90, color='green', linestyle='--', linewidth=1, alpha=0.6)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right', fontsize=8)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f'{acc:.1f}%', ha='center', fontweight='bold', fontsize=8)

ax = axes[2]
feat_corr = pd.DataFrame(X_tr, columns=FEATURES)
feat_corr['Demand'] = y_tr
corr = feat_corr.corr()['Demand'].drop('Demand').sort_values()
ax.barh(corr.index[-12:], corr.values[-12:], color=COLORS['gb'], alpha=0.8)
ax.set_title('Top 12 Feature Correlation', fontweight='bold', fontsize=11)
ax.tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.savefig('02_model_comparison.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Comparison → 02_model_comparison.png')

## Cell 7 — Simpan Model GB (kompatibel dengan `ml/app.py`)

File-file ini bisa langsung di-copy ke folder `ml/` project dan di-load oleh
`ModelManager` di `app.py` — jumlah fitur sudah konsisten 33-33-33.

In [ ]:
with open('gb_demand.pkl', 'wb') as f: pickle.dump(gb_demand, f)
with open('gb_profit.pkl', 'wb') as f: pickle.dump(gb_profit, f)
np.save('feat_mean.npy', X_MEAN)
np.save('feat_std.npy', X_STD)
np.save('feature_names.npy', np.array(FEATURES))

print('✅ Tersimpan:')
for fn in ['gb_demand.pkl', 'gb_profit.pkl', 'feat_mean.npy', 'feat_std.npy', 'feature_names.npy']:
    print(f'   {fn:<22} {os.path.getsize(fn):>10,} bytes')
print(f'\n   feat_mean.npy shape = {X_MEAN.shape}  (harus (33,) — cocok dgn _build_row() di app.py)')

## Cell 8 — Pure NumPy LSTM (eksplorasi)

**Catatan jujur:** implementasi LSTM di bawah melatih *hanya* output
projection layer (`Wy`/`by`) dengan Adam + numerical gradient — gate LSTM
(`Wf/Wi/Wg/Wo`) tetap random-init (gaya reservoir/echo-state). Ini eksplorasi
pendekatan deep-learning murni NumPy, **bukan** model yang di-serve live
(yang live cuma Gradient Boosting di atas). Dilatih di sini pakai agregat
demand harian **asli** dari CSV, bukan data sintetis.

In [ ]:
N_EPOCHS_LSTM = 80   # naikkan ke 150-200 untuk hasil lebih baik (lebih lama)
HIDDEN_LSTM   = 64
LOOKBACK      = 14

class LSTMCell:
    def __init__(self, input_size, hidden_size, seed=42):
        np.random.seed(seed)
        scale, concat = np.sqrt(2.0 / (input_size + hidden_size)), input_size + hidden_size
        self.Wf = np.random.randn(concat, hidden_size) * scale
        self.Wi = np.random.randn(concat, hidden_size) * scale
        self.Wg = np.random.randn(concat, hidden_size) * scale
        self.Wo = np.random.randn(concat, hidden_size) * scale
        self.bf = np.zeros((1, hidden_size)); self.bi = np.zeros((1, hidden_size))
        self.bg = np.zeros((1, hidden_size)); self.bo = np.zeros((1, hidden_size))
        self.h = np.zeros((1, hidden_size)); self.c = np.zeros((1, hidden_size))

    @staticmethod
    def sigmoid(x): return 1.0 / (1.0 + np.exp(-np.clip(x, -10, 10)))

    def forward(self, x):
        combined = np.hstack([x, self.h])
        f = self.sigmoid(combined @ self.Wf + self.bf)
        i = self.sigmoid(combined @ self.Wi + self.bi)
        g = np.tanh(combined @ self.Wg + self.bg)
        o = self.sigmoid(combined @ self.Wo + self.bo)
        self.c = f * self.c + i * g
        self.h = o * np.tanh(self.c)
        return self.h

    def reset(self, batch_size=1):
        self.h = np.zeros((batch_size, self.h.shape[1])); self.c = np.zeros((batch_size, self.c.shape[1]))


class LSTMForecaster:
    def __init__(self, input_size=1, hidden_size=64, seed=42):
        self.cell = LSTMCell(input_size, hidden_size, seed)
        np.random.seed(seed)
        self.Wy = np.random.randn(hidden_size, 1) * np.sqrt(2.0 / hidden_size)
        self.by = np.zeros((1, 1)); self.hidden_size = hidden_size

    def forward(self, X):
        self.cell.reset(X.shape[0])
        h = None
        for t in range(X.shape[1]):
            h = self.cell.forward(X[:, t, :])
        return h @ self.Wy + self.by

    def predict(self, X): return self.forward(X)

    def save(self, path, mean=0.0, std=1.0):
        np.savez(path, Wf=self.cell.Wf, Wi=self.cell.Wi, Wg=self.cell.Wg, Wo=self.cell.Wo,
                  bf=self.cell.bf, bi=self.cell.bi, bg=self.cell.bg, bo=self.cell.bo,
                  Wy=self.Wy, by=self.by, mean=np.array([mean]), std=np.array([std]))
        print(f'[LSTM] Saved → {path}')

print('✅ LSTMForecaster didefinisikan')

In [ ]:
print('[ML] Aggregating daily demand dari CSV (real data, bukan sintetis)...')
daily = df.groupby('Date')['Actual_Demand'].sum().sort_index()
series = daily.values.astype(np.float64)
d_mean, d_std = series.mean(), series.std() + 1e-8
norm = (series - d_mean) / d_std

Xl, yl = [], []
for i in range(len(norm) - LOOKBACK):
    Xl.append(norm[i:i + LOOKBACK]); yl.append(norm[i + LOOKBACK])
Xl = np.array(Xl).reshape(-1, LOOKBACK, 1)
yl = np.array(yl).reshape(-1, 1)

n = len(Xl)
t1, t2 = int(n * 0.70), int(n * 0.85)
Xl_tr, yl_tr = Xl[:t1], yl[:t1]
Xl_val, yl_val = Xl[t1:t2], yl[t1:t2]
Xl_te, yl_te = Xl[t2:], yl[t2:]
print(f'[ML] {len(daily)} hari → train={len(Xl_tr)}  val={len(Xl_val)}  test={len(Xl_te)} sequences')

In [ ]:
def lstm_grad(model, X, y, eps=1e-4):
    grads = {}
    for name, param in [('Wy', model.Wy), ('by', model.by)]:
        g = np.zeros_like(param)
        it = np.nditer(param, flags=['multi_index'])
        while not it.finished:
            idx = it.multi_index; orig = param[idx]
            param[idx] = orig + eps; lp = float(np.mean((model.forward(X) - y) ** 2))
            param[idx] = orig - eps; lm = float(np.mean((model.forward(X) - y) ** 2))
            param[idx] = orig
            g[idx] = (lp - lm) / (2 * eps)
            it.iternext()
        grads[name] = g
    return grads

lstm = LSTMForecaster(input_size=1, hidden_size=HIDDEN_LSTM)
m_Wy = v_Wy = np.zeros_like(lstm.Wy); m_by = v_by = np.zeros_like(lstm.by)
b1, b2, eps_a, lr = 0.9, 0.999, 1e-8, 0.001
best_val, best_w, no_improve = float('inf'), None, 0
train_losses, val_losses = [], []

print(f'[ML] Training LSTM ({HIDDEN_LSTM} units, {N_EPOCHS_LSTM} epochs, Adam)...')
t0 = time.time()
for epoch in range(1, N_EPOCHS_LSTM + 1):
    pred = lstm.forward(Xl_tr)
    loss = float(np.mean((pred - yl_tr) ** 2))
    grads = lstm_grad(lstm, Xl_tr, yl_tr)
    norm_g = np.sqrt(sum(np.sum(g ** 2) for g in grads.values()))
    if norm_g > 1.0:
        grads = {k: g / norm_g for k, g in grads.items()}
    m_Wy = b1 * m_Wy + (1 - b1) * grads['Wy']; v_Wy = b2 * v_Wy + (1 - b2) * grads['Wy'] ** 2
    lstm.Wy -= lr * (m_Wy / (1 - b1 ** epoch)) / (np.sqrt(v_Wy / (1 - b2 ** epoch)) + eps_a)
    m_by = b1 * m_by + (1 - b1) * grads['by']; v_by = b2 * v_by + (1 - b2) * grads['by'] ** 2
    lstm.by -= lr * (m_by / (1 - b1 ** epoch)) / (np.sqrt(v_by / (1 - b2 ** epoch)) + eps_a)

    val_loss = float(np.mean((lstm.predict(Xl_val) - yl_val) ** 2))
    train_losses.append(loss); val_losses.append(val_loss)
    if val_loss < best_val - 1e-6:
        best_val, best_w, no_improve = val_loss, {'Wy': lstm.Wy.copy(), 'by': lstm.by.copy()}, 0
    else:
        no_improve += 1
    if no_improve and no_improve % 15 == 0: lr *= 0.5
    if epoch % 20 == 0:
        print(f'   Epoch {epoch:3d}/{N_EPOCHS_LSTM}  train={loss:.5f}  val={val_loss:.5f}  ({time.time()-t0:.0f}s elapsed)')

if best_w: lstm.Wy, lstm.by = best_w['Wy'], best_w['by']
print(f'✅ LSTM training selesai ({time.time()-t0:.0f}s) — best val loss={best_val:.5f}')

In [ ]:
pred_norm = lstm.predict(Xl_te)
pred = pred_norm * d_std + d_mean
actual = yl_te * d_std + d_mean
lstm_mape = float(np.mean(np.abs((actual - pred) / (np.abs(actual) + 1e-8))) * 100)
lstm_acc = max(0, 100 - lstm_mape)
print(f'[ML] LSTM Test — MAPE={lstm_mape:.1f}%  Accuracy={lstm_acc:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(train_losses, label='Train Loss', color=COLORS['gb'])
axes[0].plot(val_losses, label='Val Loss', color=COLORS['ma'])
axes[0].set_title('LSTM Training Curve', fontweight='bold'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(actual[-60:], color='#1e293b', lw=2, label='Actual')
axes[1].plot(pred[-60:], color=COLORS['lstm'], lw=1.5, linestyle='--', label=f'LSTM Pred ({lstm_acc:.1f}%)')
axes[1].set_title('LSTM Forecast — 60 Hari Terakhir (Test Set)', fontweight='bold')
axes[1].set_ylabel('Total Demand (semua produk)/hari'); axes[1].legend()

plt.tight_layout()
plt.savefig('03_lstm_forecast.png', dpi=130, bbox_inches='tight')
plt.show()

lstm.save('lstm_weights.npz', mean=d_mean, std=d_std)

## Cell 9 — Sanity Test (load ulang artefak, simulasikan ModelManager)

Verifikasi cepat: artefak yang baru disimpan bisa di-load lagi dan dipakai
predict tanpa error shape — ini exact yang akan dilakukan `ModelManager`
di `app.py` saat startup.

In [ ]:
with open('gb_demand.pkl', 'rb') as f: _gb_d = pickle.load(f)
with open('gb_profit.pkl', 'rb') as f: _gb_p = pickle.load(f)
_mean, _std = np.load('feat_mean.npy'), np.load('feat_std.npy')
_names = np.load('feature_names.npy', allow_pickle=True)

assert _gb_d.n_features_in_ == len(_names) == _mean.shape[0] == _std.shape[0], \
    f"MISMATCH! gb={_gb_d.n_features_in_}  names={len(_names)}  mean={_mean.shape[0]}  std={_std.shape[0]}"
print(f'✅ Konsisten — semua artefak {_gb_d.n_features_in_} fitur, siap dipakai app.py')

sample_row = X_te[0]
X_in = ((sample_row - _mean) / _std).reshape(1, -1)
pred_demand = float(_gb_d.predict(X_in)[0])
pred_profit = float(_gb_p.predict(X_in)[0])
print(f'   Sample prediction → demand={pred_demand:.1f} units/day, gross_profit=${pred_profit:.2f}/day')
print(f'   (actual saat itu: demand={y_te[0]:.1f}, gross_profit=${yg_te[0]:.2f})')

## Cell 10 — Selesai

**File yang dihasilkan** (download semua, taruh di folder `ml/` project):
- `gb_demand.pkl`, `gb_profit.pkl` — model Gradient Boosting (live)
- `feat_mean.npy`, `feat_std.npy`, `feature_names.npy` — scaler & nama fitur
- `lstm_weights.npz` — bobot LSTM (eksplorasi, opsional)
- `01_eda_overview.png`, `02_model_comparison.png`, `03_lstm_forecast.png` — visualisasi

**Deploy:** copy 5 file pertama ke folder `ml/`, lalu jalankan:
```bash
python3 ml/app.py
```
Flask API akan otomatis memuat model baru ini di `http://localhost:5002`.

In [ ]:
print('🎉 Notebook selesai!')
print(f"\nRingkasan akurasi:")
print(f"  Gradient Boosting (demand) : {results['gb']['acc']:.1f}%")
print(f"  Gradient Boosting (profit) : {res_profit['acc']:.1f}%")
print(f"  Random Forest (pembanding) : {results['rf']['acc']:.1f}%")
print(f"  MLP (pembanding)           : {results['mlp']['acc']:.1f}%")
print(f"  Pure NumPy LSTM            : {lstm_acc:.1f}%")